<a href="https://colab.research.google.com/github/Alime21/NLP_Author_Detection/blob/main/Alime_K%C4%B1l%C4%B1n%C3%A7_220709074.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ==========================================
# CELL 1: LIBRARIES AND DRIVE INSTALLATION
# ==========================================
# 1. Connect Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Main Directory path
import os
base_path = '/content/drive/MyDrive/nlp_project'

# 3. Making folders
for folder in ['data', 'vocab', 'models']:
    os.makedirs(f'{base_path}/{folder}', exist_ok=True)


Mounted at /content/drive


In [4]:
# ==========================================
# CELL 2:  DATA DOWNLOAD AND CLEANING
# ==========================================
import requests
import re
import pickle
import nltk
import nltk.internals
from nltk.tokenize import word_tokenize
from collections import defaultdict

# NLTK packets
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

AUTHORS = {
    "Mark Twain": [74, 76, 1837, 86, 245],
    "Jane Austen": [1342, 161, 158, 121, 105],
    "Arthur Conan Doyle": [2852, 244, 2097, 1661, 834],
    "H.G. Wells": [35, 36, 5230, 159, 1013]
}

def get_clean_text(book_id):
  """Download the book from Gutenberg and delete the license text."""
  url = f"https://www.gutenberg.org/cache/epub/{book_id}/pg{book_id}.txt"
  resp = requests.get(url)
  resp.encoding = 'utf-8'
  text = resp.text

  start = re.search(r'\*\*\* START OF THE PROJECT GUTENBERG.*?\*\*\*', text)
  end = re.search(r'\*\*\* END OF THE PROJECT GUTENBERG.*?\*\*\*', text)
  if start and end:
        text = text[start.end():end.start()]
  return text

def chunk_text(text, chunk_size=500):
  """Remove punctuation from the text and divide it into 500-word blocks"""
  tokens = [w for w in word_tokenize(text.lower()) if w.isalnum()]
  chunks = []
  for i in range(0, len(tokens), chunk_size):
    chunk = tokens[i:i+chunk_size]
    if len(chunk) == chunk_size:
      chunks.append(chunk)
  return chunks

base_path = '/content/drive/MyDrive/nlp_project'
dataset = defaultdict(list)

for author, ids in AUTHORS.items():
  print(f"-> {author} books are downloading")
  for b_id in ids:
    raw_text = get_clean_text(b_id)
    dataset[author].extend(chunk_text(raw_text))

  dataset[author] = dataset[author][:200]
  print(f"   [{author}] completed! Sum {len(dataset[author])} track is ready.")

# Save all raw data to the passages.pkl file
with open(f'{base_path}/data/passages.pkl', 'wb') as f:
  pickle.dump(dataset, f)

print("\n[SUCCESS] All texts have been downloaded, tokenized, and saved to Drive as 'data/passages.pkl'!")



-> Mark Twain books are downloading
   [Mark Twain] completed! Sum 200 track is ready.
-> Jane Austen books are downloading
   [Jane Austen] completed! Sum 200 track is ready.
-> Arthur Conan Doyle books are downloading
   [Arthur Conan Doyle] completed! Sum 200 track is ready.
-> H.G. Wells books are downloading
   [H.G. Wells] completed! Sum 200 track is ready.

[SUCCESS] All texts have been downloaded, tokenized, and saved to Drive as 'data/passages.pkl'!


In [5]:
# ==========================================
# CELL 3: DATA SPLITTING (STRATIFIED SPLIT)
# ==========================================
import pickle
import random

base_path = '/content/drive/MyDrive/nlp_project'
with open(f'{base_path}/data/passages.pkl', 'rb') as f:
  dataset = pickle.load(f)

random.seed(42)

train_corpus, train_labels = [], []
val_corpus, val_labels = [], []
test_corpus, test_labels = [], []

for author, passages in dataset.items():
  # Randomly shuffle 200 texts by each author within themselves
  random.shuffle(passages)

  # Proportions of the 200 pieces: 140 (70%), 20 (10%), 40 (20%)
  train_end = 140
  val_end = 160

  train_passages = passages[:train_end]
  val_passages = passages[train_end:val_end]
  test_passages = passages[val_end:]

  # Add Train lists
  train_corpus.extend(train_passages)
  train_labels.extend([author] * len(train_passages))

  # Add Val lists
  val_corpus.extend(val_passages)
  val_labels.extend([author] * len(val_passages))

  # Add Test lists
  test_corpus.extend(test_passages)
  test_labels.extend([author] * len(test_passages))

# To prevent the AI writers from memorizing the sets in order, shuffle all sets with their authors
def shuffle_together(corpus, labels):
    combined = list(zip(corpus, labels))
    random.shuffle(combined)
    return [c[0] for c in combined], [c[1] for c in combined]

train_corpus, train_labels = shuffle_together(train_corpus, train_labels)
val_corpus, val_labels = shuffle_together(val_corpus, val_labels)
test_corpus, test_labels = shuffle_together(test_corpus, test_labels)


with open(f'{base_path}/data/train.pkl', 'wb') as f:
    pickle.dump((train_corpus, train_labels), f)

with open(f'{base_path}/data/val.pkl', 'wb') as f:
    pickle.dump((val_corpus, val_labels), f)

with open(f'{base_path}/data/test.pkl', 'wb') as f:
    pickle.dump((test_corpus, test_labels), f)

print("\n[SUCCESS] Stratified Split completed!")
print(f"Training Set: {len(train_corpus)} items (Expected: 560)")
print(f"Validation Set: {len(val_corpus)} pieces (Expected: 80)")
print(f"Test Set: {len(test_corpus)} pieces (Expected: 160)")




[SUCCESS] Stratified Split completed!
Training Set: 560 items (Expected: 560)
Validation Set: 80 pieces (Expected: 80)
Test Set: 160 pieces (Expected: 160)


In [6]:
# ==========================================
# CELL 4: VSM (VECTOR SPACE MODEL) TRAINING
# ==========================================
import pickle
import math
import time
import numpy as np
from collections import Counter

base_path = '/content/drive/MyDrive/nlp_project'

print("1. Training data (Train) is being read...")
with open(f'{base_path}/data/train.pkl', 'rb') as f:
    train_corpus, train_labels = pickle.load(f)

# --- MATHEMATICS AND MODEL FUNCTIONS ---
def build_vocab(corpus):
  """Creates a dictionary from unique words in the corpus."""
  word2idx = {}
  for doc in corpus:
    for word in doc:
      if word not in word2idx:
        word2idx[word] = len(word2idx)
  return word2idx

def compute_idf(corpus, word2idx):
  """Calculates IDF (Inverse Document Frequency). Rare words receive higher scores."""
  N = len(corpus)
  df = Counter()
  for doc in corpus:
    for word in set(doc):
      if word in word2idx:
        df[word] += 1

  idf = {}
  for word, index in word2idx.items():
        count = df.get(word, 0)
        idf[word] = math.log(N / (count + 1))
  return idf

def tfidf_vectorize(doc,word2idx, idf):
    """Transforms a text into a numerical vector with TF-IDF weights."""
    vec = np.zeros(len(word2idx))
    term_counts = Counter(doc)

    for word, count in term_counts.items():
      if word in word2idx:
        idx = word2idx[word]
        vec[idx] = count * idf[word]
    return vec

 # ---------- TRAINING ------------
print("2. The dictionary (vocabulary) and IDF values are calculated...")
start_time = time.time()

word2idx = build_vocab(train_corpus)
print(f"   -> Dictionary size: {len(word2idx)} unique words found.")

idf = compute_idf(train_corpus, word2idx)

print("3. The author has center points (Centroids)...")
centroids = {label: np.zeros(len(word2idx)) for label in set(train_labels)}
class_counts = Counter(train_labels)

 # Place all texts in space and add them to the center of their author
for doc, label in zip(train_corpus, train_labels):
    vec = tfidf_vectorize(doc, word2idx, idf)
    centroids[label] += vec

 # Find the average (centroid) by dividing the total vectors by the number of texts
for label in centroids:
    centroids[label] /= class_counts[label]

train_time = time.time() - start_time
print(f"   -> Training completed! (Duration: {train_time:.2f} seconds)")

 # --- SAVING STAGE ---
print("4. Saving to Model Drive...")
# Save the vocabulary
with open(f'{base_path}/vocab/word2idx.pkl', 'wb') as f:
    pickle.dump(word2idx, f)

# Save the VSM Model (Centroids and IDF)
vsm_model_data = {'centroids': centroids, 'idf': idf}
with open(f'{base_path}/models/vsm.pkl', 'wb') as f:
    pickle.dump(vsm_model_data, f)

print("\n[SUCCESSFUL] VSM Model (vsm.pkl) and Dictionary (word2idx.pkl) were saved successfully!")




1. Training data (Train) is being read...
2. The dictionary (vocabulary) and IDF values are calculated...
   -> Dictionary size: 14862 unique words found.
3. The author has center points (Centroids)...
   -> Training completed! (Duration: 0.22 seconds)
4. Saving to Model Drive...

[SUCCESSFUL] VSM Model (vsm.pkl) and Dictionary (word2idx.pkl) were saved successfully!


In [7]:
# ==========================================
# CELL 5: VSM MODEL TESTING AND EVALUATION
# ==========================================
import pickle
import numpy as np
from collections import Counter

base_path = '/content/drive/MyDrive/nlp_project'

print("1. Loading Trained VSM Model and Test Data...")
with open(f'{base_path}/data/test.pkl', 'rb') as f:
    test_corpus, test_labels = pickle.load(f)

with open(f'{base_path}/vocab/word2idx.pkl', 'rb') as f:
    word2idx = pickle.load(f)

with open(f'{base_path}/models/vsm.pkl', 'rb') as f:
    vsm_model = pickle.load(f)

centroids = vsm_model['centroids']
idf = vsm_model['idf']

# --- MATHEMATICAL PREDICTION FUNCTIONS ---
def tfidf_vectorize(doc, word2idx, idf):
    vec = np.zeros(len(word2idx))
    term_counts = Counter(doc)
    for word, count in term_counts.items():
        if word in word2idx:
            vec[word2idx[word]] = count * idf[word]
    return vec

def cosine_similarity(v1, v2):
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 == 0 or n2 == 0: return 0.0
    return np.dot(v1, v2) / (n1 * n2)

def predict_author(doc, word2idx, idf, centroids):
    doc_vec = tfidf_vectorize(doc, word2idx, idf)
    best_author = None
    max_sim = -float('inf')

    for author, centroid_vec in centroids.items():
        sim = cosine_similarity(doc_vec, centroid_vec)
        if sim > max_sim:
            max_sim = sim
            best_author = author
    return best_author

print("2. Running the Model Evaluation (This may take a few seconds)...")
correct_preds = 0
total_docs = len(test_labels)

# Query the machine for each text in the test set
for doc, true_label in zip(test_corpus, test_labels):
    predicted = predict_author(doc, word2idx, idf, centroids)
    if predicted == true_label:
        correct_preds += 1

accuracy = (correct_preds / total_docs) * 100

print("==========================================")
print(f"VSM MODEL TEST RESULTS")
print(f"Total Questions Asked: {total_docs}")
print(f"Correct Predictions: {correct_preds}")
print(f"ACCURACY RATE: %{accuracy:.2f}")

1. Loading Trained VSM Model and Test Data...
2. Running the Model Evaluation (This may take a few seconds)...
VSM MODEL TEST RESULTS
Total Questions Asked: 160
Correct Predictions: 155
ACCURACY RATE: %96.88


In [8]:
# ==========================================
# CELL 6: N-GRAM LANGUAGE MODEL (TRAINING)
# ==========================================
import pickle
import time
from collections import defaultdict, Counter

base_path = '/content/drive/MyDrive/nlp_project'

print("1. Loading training data...")
with open(f'{base_path}/data/train.pkl', 'rb') as f:
    train_corpus, train_labels = pickle.load(f)

# --- 1. CREATING LANGUAGE MODEL VOCABULARY (LM VOCAB) ---
# To prevent the model from crashing on unseen words, we replace
# rare words with the "<UNK>" (Unknown) token.
print("2. Creating Special Vocabulary (LM Vocab) for N-Gram...")
word_freqs = Counter()
for doc in train_corpus:
    word_freqs.update(doc)

# Include only words that appear more than twice; discard the rest
min_freq = 2
lm_vocab = {word for word, freq in word_freqs.items() if freq >= min_freq}
lm_vocab.add("<UNK>") # Lifeboat for unknown words
print(f"   -> Original word count: {len(word_freqs)}")
print(f"   -> LM Vocab Size (including <UNK>): {len(lm_vocab)} words.")

def replace_unk(doc, vocab):
    """Replaces words not in the vocabulary with the <UNK> token."""
    return [w if w in vocab else "<UNK>" for w in doc]

# --- 2. N-GRAM (BIGRAM) MODEL TRAINING ---
print("3. Training Bigram (Word Pairs) Models for Authors...")
start_time = time.time()

# Unigram (single word) and Bigram (word pair) counters for each author
# Using defaultdict to automatically initialize counters for new authors
unigram_counts = defaultdict(Counter)
bigram_counts = defaultdict(Counter)

for doc, author in zip(train_corpus, train_labels):
    # First, replace rare words in the text with <UNK>
    processed_doc = replace_unk(doc, lm_vocab)

    # Count words individually (unigram) and in pairs (bigram)
    for i in range(len(processed_doc)):
        unigram_counts[author][processed_doc[i]] += 1

        # If not at the last word, capture the pair (bigram)
        if i < len(processed_doc) - 1:
            bigram = (processed_doc[i], processed_doc[i+1])
            bigram_counts[author][bigram] += 1

train_time = time.time() - start_time
print(f"   -> Training completed successfully! (Duration: {train_time:.2f} seconds)")

# --- 3. SAVING MODELS ---
print("4. Saving N-Gram Model to Drive...")

# Save the lm_vocab file as required
with open(f'{base_path}/vocab/lm_vocab.pkl', 'wb') as f:
    pickle.dump(lm_vocab, f)

# Save the counters to be used in probability calculations
ngram_model_data = {
    'unigram_counts': unigram_counts,
    'bigram_counts': bigram_counts,
    'vocab_size': len(lm_vocab)
}
with open(f'{base_path}/models/ngram.pkl', 'wb') as f:
    pickle.dump(ngram_model_data, f)

print("\n[SUCCESS] N-Gram Model (ngram.pkl) and Vocabulary (lm_vocab.pkl) saved to Drive!")

1. Loading training data...
2. Creating Special Vocabulary (LM Vocab) for N-Gram...
   -> Original word count: 14862
   -> LM Vocab Size (including <UNK>): 8724 words.
3. Training Bigram (Word Pairs) Models for Authors...
   -> Training completed successfully! (Duration: 0.46 seconds)
4. Saving N-Gram Model to Drive...

[SUCCESS] N-Gram Model (ngram.pkl) and Vocabulary (lm_vocab.pkl) saved to Drive!


In [9]:
# ==========================================
# CELL 7: N-GRAM MODEL TESTING AND EVALUATION
# ==========================================
import pickle
import math

base_path = '/content/drive/MyDrive/nlp_project'

print("1. Loading Trained N-Gram Model and Test Data...")
with open(f'{base_path}/data/test.pkl', 'rb') as f:
    test_corpus, test_labels = pickle.load(f)

with open(f'{base_path}/vocab/lm_vocab.pkl', 'rb') as f:
    lm_vocab = pickle.load(f)

with open(f'{base_path}/models/ngram.pkl', 'rb') as f:
    ngram_model = pickle.load(f)

unigram_counts = ngram_model['unigram_counts']
bigram_counts = ngram_model['bigram_counts']
vocab_size = ngram_model['vocab_size']
authors = list(unigram_counts.keys())

# --- MATHEMATICAL PREDICTION FUNCTION (LAPLACE SMOOTHING) ---
def calculate_log_prob(doc, author):
    """Calculates the probability (Log-Prob) of a text being written by a specific author."""
    # Replace rare words not in vocabulary with <UNK>
    processed_doc = [w if w in lm_vocab else "<UNK>" for w in doc]

    log_prob = 0.0
    # Since probabilities are very small, we use Logarithmic Addition instead of multiplication
    for i in range(len(processed_doc) - 1):
        w1 = processed_doc[i]
        w2 = processed_doc[i+1]
        bigram = (w1, w2)

        # LAPLACE (ADD-1) SMOOTHING FORMULA: (Bigram Count + 1) / (Unigram Count + Vocab Size)
        bg_count = bigram_counts[author].get(bigram, 0)
        ug_count = unigram_counts[author].get(w1, 0)

        prob = (bg_count + 1) / (ug_count + vocab_size)
        log_prob += math.log(prob) # Sum the logarithms

    return log_prob

def predict_author_ngram(doc):
    """Finds the author with the highest probability for the given text."""
    best_author = None
    max_log_prob = -float('inf') # Start with negative infinity

    for author in authors:
        score = calculate_log_prob(doc, author)
        if score > max_log_prob:
            max_log_prob = score
            best_author = author
    return best_author

print("2. Putting the Machine to the Test (Calculating N-Gram Probabilities)...")
y_true = test_labels
y_pred = []

correct_preds = 0
total_docs = len(test_labels)

for doc, true_label in zip(test_corpus, y_true):
    predicted = predict_author_ngram(doc)
    y_pred.append(predicted)
    if predicted == true_label:
        correct_preds += 1

accuracy = (correct_preds / total_docs) * 100

print("\n==========================================")
print(f"N-GRAM MODEL TEST RESULTS")
print(f"Total Questions Asked: {total_docs}")
print(f"Correct Predictions: {correct_preds}")
print(f"ACCURACY RATE: %{accuracy:.2f}")
print("==========================================")

# Confusion Matrix for a fair comparison with VSM
cm = {actual: {pred: 0 for pred in authors} for actual in authors}
for actual, pred in zip(y_true, y_pred):
    cm[actual][pred] += 1

print("\n CONFUSION MATRIX")
print(f"{'Actual \\ Predicted':<20}", "".join([f"{y[:10]:<12}" for y in authors]))
print("-" * 65)
for actual in authors:
    row = f"{actual[:15]:<20}"
    for pred in authors:
        row += f"{cm[actual][pred]:<12}"
    print(row)

1. Loading Trained N-Gram Model and Test Data...
2. Putting the Machine to the Test (Calculating N-Gram Probabilities)...

N-GRAM MODEL TEST RESULTS
Total Questions Asked: 160
Correct Predictions: 158
ACCURACY RATE: %98.75

 CONFUSION MATRIX
Actual \ Predicted   Arthur Con  Mark Twain  Jane Auste  H.G. Wells  
-----------------------------------------------------------------
Arthur Conan Do     40          0           0           0           
Mark Twain          0           39          0           1           
Jane Austen         0           0           40          0           
H.G. Wells          0           1           0           39          


In [10]:
# ==========================================
# CELL 8: NEURAL NETWORK (MLP) TRAINING (PYTORCH)
# ==========================================
import torch
import torch.nn as nn
import torch.optim as optim
import pickle
import numpy as np
import time
from collections import Counter

base_path = '/content/drive/MyDrive/nlp_project'

print("1. Loading Training Data and VSM Vocabulary...")
with open(f'{base_path}/data/train.pkl', 'rb') as f:
    train_corpus, train_labels = pickle.load(f)

# We will use the vocabulary and IDF values created in VSM for the MLP
with open(f'{base_path}/vocab/word2idx.pkl', 'rb') as f:
    word2idx = pickle.load(f)

with open(f'{base_path}/models/vsm.pkl', 'rb') as f:
    vsm_data = pickle.load(f)
    idf = vsm_data['idf']

# --- DATA PREPARATION (CONVERTING NUMBERS TO TENSORS) ---
# PyTorch only understands "Tensors," which are its own mathematical objects
print("2. Converting Texts into Tensors (Matrices) for the Neural Network...")

authors = list(set(train_labels))
label2idx = {author: idx for idx, author in enumerate(authors)} # Assign IDs (0,1,2,3) to authors

# Save the author names to remember them later
with open(f'{base_path}/models/mlp_labels.pkl', 'wb') as f:
    pickle.dump(label2idx, f)

def build_tensors(corpus, labels, word2idx, idf, label2idx):
    """Creates a massive matrix (X) where rows are documents and columns are words, along with labels (Y)"""
    X = np.zeros((len(corpus), len(word2idx)), dtype=np.float32)
    Y = np.zeros(len(corpus), dtype=np.int64)

    for i, doc in enumerate(corpus):
        term_counts = Counter(doc)
        for word, count in term_counts.items():
            if word in word2idx:
                X[i, word2idx[word]] = count * idf[word] # TF-IDF value
        Y[i] = label2idx[labels[i]]

    return torch.tensor(X), torch.tensor(Y)

X_train, y_train = build_tensors(train_corpus, train_labels, word2idx, idf, label2idx)

# --- NEURAL NETWORK ARCHITECTURE (THE BRAIN) ---
class AuthorClassifierMLP(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(AuthorClassifierMLP, self).__init__()
        # Layer 1 (Input -> Hidden Layer)
        self.layer1 = nn.Linear(input_size, hidden_size)
        # Activation Function (The firing threshold of the neuron in the brain)
        self.relu = nn.ReLU()
        # Layer 2 (Hidden Layer -> Output/Prediction)
        self.layer2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out = self.layer1(x)
        out = self.relu(out)
        out = self.layer2(out)
        return out

input_size = len(word2idx)
hidden_size = 128  # Number of neurons in the intermediate layer
num_classes = len(authors) # 4 authors

model = AuthorClassifierMLP(input_size, hidden_size, num_classes)

# Loss Function and Optimizer (The Teacher)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# --- TRAINING LOOP ---
print("3. Training the Neural Network (Epochs Starting)...")
start_time = time.time()
num_epochs = 30 # The dataset will be iterated over 30 times

for epoch in range(num_epochs):
    # Forward pass
    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    # Backpropagation and Optimization (Neurons learning from error)
    optimizer.zero_grad() # Reset previous gradients
    loss.backward()       # Calculate gradients backward
    optimizer.step()      # Update weights (neuron connections)

    # Provide status update every 5 steps
    if (epoch+1) % 5 == 0:
        print(f'   -> Epoch [{epoch+1}/{num_epochs}], Loss (Error Rate): {loss.item():.4f}')

train_time = time.time() - start_time
print(f"\n[TRAINING FINISHED] Duration: {train_time:.2f} seconds")

# --- SAVING THE MODEL ---
print("4. Saving the Trained 'Brain' as 'mlp.pt' to Drive...")
# PyTorch models are saved with .pt or .pth extensions. state_dict() stores the weights.
torch.save(model.state_dict(), f'{base_path}/models/mlp.pt')

print(f"\n[SUCCESS] The requested file 'mlp.pt' has been successfully created!")

1. Loading Training Data and VSM Vocabulary...
2. Converting Texts into Tensors (Matrices) for the Neural Network...
3. Training the Neural Network (Epochs Starting)...
   -> Epoch [5/30], Loss (Error Rate): 0.0000
   -> Epoch [10/30], Loss (Error Rate): 0.0000
   -> Epoch [15/30], Loss (Error Rate): 0.0000
   -> Epoch [20/30], Loss (Error Rate): 0.0000
   -> Epoch [25/30], Loss (Error Rate): 0.0000
   -> Epoch [30/30], Loss (Error Rate): 0.0000

[TRAINING FINISHED] Duration: 4.01 seconds
4. Saving the Trained 'Brain' as 'mlp.pt' to Drive...

[SUCCESS] The requested file 'mlp.pt' has been successfully created!


In [12]:
# ==========================================
# CELL 9: NEURAL NETWORK (MLP) TESTING AND EVALUATION
# ==========================================
import torch
import torch.nn as nn
import pickle
import numpy as np
from collections import Counter

base_path = '/content/drive/MyDrive/nlp_project'

print("1. Loading Test Data and the Sleeping Brain (mlp.pt)...")
with open(f'{base_path}/data/test.pkl', 'rb') as f:
    test_corpus, test_labels = pickle.load(f)

with open(f'{base_path}/vocab/word2idx.pkl', 'rb') as f:
    word2idx = pickle.load(f)

with open(f'{base_path}/models/vsm.pkl', 'rb') as f:
    vsm_data = pickle.load(f)
    idf = vsm_data['idf']

with open(f'{base_path}/models/mlp_labels.pkl', 'rb') as f:
    label2idx = pickle.load(f)

# Dictionary to convert IDs back to author names (e.g., 0 -> Mark Twain)
idx2label = {idx: label for label, idx in label2idx.items()}
authors = list(label2idx.keys())

print("2. Converting Exam Papers to Numbers (Tensors)...")
def build_test_tensors(corpus, labels, word2idx, idf, label2idx):
    X = np.zeros((len(corpus), len(word2idx)), dtype=np.float32)
    Y = np.zeros(len(corpus), dtype=np.int64)
    for i, doc in enumerate(corpus):
        term_counts = Counter(doc)
        for word, count in term_counts.items():
            if word in word2idx:
                X[i, word2idx[word]] = count * idf[word]
        Y[i] = label2idx[labels[i]]
    return torch.tensor(X), torch.tensor(Y)

X_test, y_test = build_test_tensors(test_corpus, test_labels, word2idx, idf, label2idx)

# --- REBUILDING THE NEURAL NETWORK ---
print("3. Waking Up the Neural Network and Setting to Evaluation Mode...")
class AuthorClassifierMLP(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(AuthorClassifierMLP, self).__init__()
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out = self.layer1(x)
        out = self.relu(out)
        out = self.layer2(out)
        return out

input_size = len(word2idx)
hidden_size = 128
num_classes = len(authors)

# Set up the skeleton of the machine
model = AuthorClassifierMLP(input_size, hidden_size, num_classes)
# Load the trained neuron connections (mlp.pt) into the skeleton
model.load_state_dict(torch.load(f'{base_path}/models/mlp.pt'))

# CRITICAL: We are switching the machine from "Training" mode to "Evaluation" (Test) mode.
# This prevents it from trying to learn or memorize during the exam.
model.eval()

# --- INFERENCE (EXAM) STAGE ---
print("4. Putting the Machine to the Test...")
with torch.no_grad(): # Disable learning (Gradients) entirely during the exam
    # Pass all test data to the machine at once
    outputs = model(X_test)
    # Get the index of the column (author) with the highest probability for each row
    _, predicted_indices = torch.max(outputs, 1)

# Count correct predictions
correct_preds = (predicted_indices == y_test).sum().item()
total_docs = len(y_test)
accuracy = (correct_preds / total_docs) * 100

print("\n==========================================")
print(f"NEURAL NETWORK (MLP) TEST RESULTS")
print(f"Total Questions Asked: {total_docs}")
print(f"Correct Predictions: {correct_preds}")
print(f"ACCURACY RATE: %{accuracy:.2f}")
print("==========================================")

# Confusion Matrix for comparison
y_true_labels = [idx2label[idx.item()] for idx in y_test]
y_pred_labels = [idx2label[idx.item()] for idx in predicted_indices]

cm = {actual: {pred: 0 for pred in authors} for actual in authors}
for actual, pred in zip(y_true_labels, y_pred_labels):
    cm[actual][pred] += 1

print("\n CONFUSION MATRIX")
print(f"{'Actual \\ Predicted':<20}", "".join([f"{y[:10]:<12}" for y in authors]))
print("-" * 65)
for actual in authors:
    row = f"{actual[:15]:<20}"
    for pred in authors:
        row += f"{cm[actual][pred]:<12}"
    print(row)

1. Loading Test Data and the Sleeping Brain (mlp.pt)...
2. Converting Exam Papers to Numbers (Tensors)...
3. Waking Up the Neural Network and Setting to Evaluation Mode...
4. Putting the Machine to the Test...

NEURAL NETWORK (MLP) TEST RESULTS
Total Questions Asked: 160
Correct Predictions: 127
ACCURACY RATE: %79.38

 CONFUSION MATRIX
Actual \ Predicted   Arthur Con  Jane Auste  Mark Twain  H.G. Wells  
-----------------------------------------------------------------
Arthur Conan Do     40          0           0           0           
Jane Austen         3           37          0           0           
Mark Twain          7           0           33          0           
H.G. Wells          23          0           0           17          
